In [ ]:
# Example 1: Using pythran.generate_cxx() to Generate C++ Code
import pythran
import tempfile
import os

# Pythran converts annotated Python code to C++
# Let's create a simple function and generate C++ code

assert pythran.__version__, "pythran.__version__ is empty"
print(f"pythran {pythran.__version__}")
print("Using pythran.generate_cxx() to Generate C++ Code:")
print("=" * 55)

# Define a simple Pythran-compatible function with annotation
pythran_code = '''
#pythran export simple_multiply(float, float)
def simple_multiply(a, b):
    """Simple multiplication function"""
    return a * b

#pythran export vector_sum(float list)
def vector_sum(values):
    """Sum all values in a list"""
    total = 0.0
    for value in values:
        total += value
    return total
'''

print("Original Python code:")
print(pythran_code)

# Generate C++ code from the Python code
print("\nGenerating C++ code with pythran.generate_cxx()...")
try:
    module_name = "example_module"
    # generate_cxx returns a tuple whose first item is the generated module; str() of that module is the C++ source.
    cxx_module, *_ = pythran.generate_cxx(module_name, pythran_code)
    cxx_code = str(cxx_module)
    
    print(f"✓ Successfully generated C++ code")
    print(f"  Module name: {module_name}")
    print(f"  Generated C++ code length: {len(cxx_code)} characters")
    
    # Show a snippet of the generated C++ code
    lines = cxx_code.split('\n')
    print(f"\nFirst 10 lines of generated C++:")
    for line in lines[:10]:
        print(f"  {line}")
    
    # Both exported functions appear in the generated C++
    for exported in ("simple_multiply", "vector_sum"):
        assert exported in cxx_code, exported
    print("\nGenerated functions found: simple_multiply, vector_sum")
    
except Exception as e:
    print(f"Error: {e}")
    raise AssertionError(
        f"pythran example (cell 0) failed: {type(e).__name__}: {e}"
    ) from e

print("\npythran.generate_cxx() transforms Python to optimized C++")

In [ ]:
# Example 2: Using pythran.compile_pythrancode() for Runtime Compilation
import importlib.util
import os
import sysconfig
import tempfile

import numpy as np
import pythran

print("Using pythran.compile_pythrancode() for Runtime Compilation:")
print("=" * 60)

numpy_code = """
#pythran export dot_product(float64[], float64[])
def dot_product(a, b):
    "Compute dot product of two vectors"
    result = 0.0
    for i in range(len(a)):
        result += a[i] * b[i]
    return result

#pythran export euclidean_distance(float64[], float64[])
def euclidean_distance(a, b):
    "Compute Euclidean distance between two points"
    sum_sq = 0.0
    for i in range(len(a)):
        diff = a[i] - b[i]
        sum_sq += diff * diff
    return sum_sq ** 0.5
"""

print("Python code to compile:")
print(numpy_code)

a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, -1.0, 0.5])

with tempfile.TemporaryDirectory() as tmpdir:
    module_name = "compiled_math"
    # output_file decides where the extension module is written; without it, it lands in the working directory
    so_path = os.path.join(tmpdir, module_name + sysconfig.get_config_var("EXT_SUFFIX"))
    result = pythran.compile_pythrancode(module_name, numpy_code, output_file=so_path)
    print(f"Compiled to: {os.path.basename(result)}")
    assert result == so_path and os.path.isfile(so_path), result

    spec = importlib.util.spec_from_file_location(module_name, so_path)
    compiled_math = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(compiled_math)

    dot = compiled_math.dot_product(a, b)
    dist = compiled_math.euclidean_distance(a, b)

print(f"dot_product(a, b)        = {dot}  (numpy: {np.dot(a, b)})")
print(f"euclidean_distance(a, b) = {dist:.6f}  (numpy: {np.linalg.norm(a - b):.6f})")
assert np.isclose(dot, np.dot(a, b)), dot
assert np.isclose(dist, np.linalg.norm(a - b)), dist

In [ ]:
# Example 2 (continued): C++-only generation (cpponly=True needs no C++ compiler)
import pythran
import tempfile
import os

# Create a more complex function with NumPy-like operations
numpy_code = '''
#pythran export dot_product(float list, float list)
def dot_product(a, b):
    """Compute dot product of two vectors"""
    result = 0.0
    for i in range(len(a)):
        result += a[i] * b[i]
    return result

#pythran export euclidean_distance(float list, float list)
def euclidean_distance(a, b):
    """Compute Euclidean distance between two points"""
    sum_sq = 0.0
    for i in range(len(a)):
        diff = a[i] - b[i]
        sum_sq += diff * diff
    return sum_sq ** 0.5
'''

# Demonstrate C++ only generation
print("\nGenerating C++ code only (cpponly=True)...")
try:
    with tempfile.TemporaryDirectory() as tmpdir:
        cpp_file = pythran.compile_pythrancode(
            "test_module",
            numpy_code,
            module_dir=tmpdir,
            output_file=os.path.join(tmpdir, "test_module.cpp"),  # keep the .cpp out of CWD (module_dir does not cover cpponly)
            cpponly=True  # Only generate C++ file
        )
        
        print(f"✓ C++ generation successful!")
        print(f"  Output: {cpp_file}")
        
        if cpp_file and os.path.exists(cpp_file):
            file_size = os.path.getsize(cpp_file)
            print(f"  File size: {file_size} bytes")
            
except Exception as e:
    print(f"Error: {e}")
    raise

print("\npythran.compile_pythrancode() enables dynamic compilation")

In [ ]:
# Example 3: Using pythran.get_include() and Configuration
import pythran
import os

print("Using pythran.get_include() and Configuration:")
print("=" * 47)

# get_include() returns the directory containing Pythran C++ headers
# This is essential when integrating Pythran with build systems

include_dir = pythran.get_include()
print(f"Pythran include directory:")
print(f"  {include_dir}")
# The C++ runtime headers live in the pythonic/ folder under the include directory
assert os.path.isdir(os.path.join(include_dir, "pythonic")), include_dir

# Check what's in the include directory
if os.path.exists(include_dir):
    items = os.listdir(include_dir)
    print(f"\nContents ({len(items)} items):")
    
    # Show subdirectories
    dirs = [item for item in items if os.path.isdir(os.path.join(include_dir, item))]
    print(f"  Subdirectories: {', '.join(dirs[:5])}")
    
    # Show some header files
    headers = [item for item in items if item.endswith('.hpp') or item.endswith('.h')]
    if headers:
        print(f"  Header files: {', '.join(headers[:5])}")

# Check Pythran configuration
print("\nPythran configuration:")
if hasattr(pythran, 'config'):
    config = pythran.config
    print(f"  Config module: {config}")
    
    # Show available config attributes
    config_attrs = [attr for attr in dir(config) if not attr.startswith('_')]
    print(f"  Available config attributes: {len(config_attrs)}")
    print(f"  Examples: {', '.join(config_attrs[:8])}")

# Check Pythran version
print(f"\nPythran version information:")
if hasattr(pythran, 'version'):
    print(f"  Version module available: Yes")
    version_attrs = [attr for attr in dir(pythran.version) if not attr.startswith('_')]
    print(f"  Version attributes: {', '.join(version_attrs)}")

# Check available intrinsics
print(f"\nPythran intrinsic functions:")
if hasattr(pythran, 'intrinsic'):
    intrinsic_module = pythran.intrinsic
    intrinsic_items = [attr for attr in dir(intrinsic_module) if not attr.startswith('_')]
    print(f"  Available intrinsics: {len(intrinsic_items)}")
    print(f"  Examples: {', '.join(intrinsic_items[:10])}")

# Check type system
print(f"\nPythran type system:")
if hasattr(pythran, 'types'):
    types_module = pythran.types
    type_items = [attr for attr in dir(types_module) if not attr.startswith('_')]
    print(f"  Available types: {len(type_items)}")
    print(f"  Examples: {', '.join(type_items[:10])}")

print("\npythran.get_include() is essential for C++ build integration")

In [ ]:
# Example 4: Using pythran.spec_parser and Load Specifications
import pythran
from pythran import spec_parser
import tempfile

print("Using pythran.spec_parser and Load Specifications:")
print("=" * 53)

# spec_parser extracts function signatures from Pythran annotations
# This is crucial for understanding what will be compiled

# Example code with multiple function signatures
code_with_specs = '''
#pythran export add(int, int)
#pythran export add(float, float)
def add(a, b):
    return a + b

#pythran export matrix_op(float[][], float)
def matrix_op(matrix, scalar):
    result = []
    for row in matrix:
        new_row = [val * scalar for val in row]
        result.append(new_row)
    return result

#pythran export process_data(float list, int)
def process_data(data, threshold):
    return [x for x in data if x > threshold]
'''

print("Python code with Pythran annotations:")
print(code_with_specs)

# Parse the specifications from the code
print("\nParsing specifications with spec_parser...")
try:
    # spec_parser returns a pythran.spec.Spec; its .functions maps each exported name to its signatures
    specs = pythran.spec_parser(code_with_specs)
    
    print(f"✓ Successfully parsed specifications")
    print(f"  Functions found: {len(specs.functions)}")
    
    # Display each function's specifications
    for func_name, func_specs in specs.functions.items():
        print(f"\n  {func_name}:")
        print(f"    Number of signatures: {len(func_specs)}")
        for i, spec in enumerate(func_specs, 1):
            print(f"    Signature {i}: {spec}")
    
    assert set(specs.functions) == {"add", "matrix_op", "process_data"}, specs.functions
    assert set(specs.functions["add"]) == {(int, int), (float, float)}, specs.functions["add"]
            
except Exception as e:
    print(f"Error: {e}")
    raise

# Test with simpler code
print("\nTesting with simpler example:")
simple_code = '''
#pythran export multiply(float, float)
def multiply(x, y):
    return x * y
'''

try:
    simple_specs = pythran.spec_parser(simple_code)
    print(f"✓ Parsed simple specifications:")
    for func, sigs in simple_specs.functions.items():
        print(f"  Function: {func}")
        print(f"  Signatures: {sigs}")
    assert simple_specs.functions == {"multiply": ((float, float),)}, simple_specs.functions
except Exception as e:
    print(f"Error: {e}")
    raise

# load_specfile reads the same export lines from a separate .pythran file
print("\nSpecification file loading:")
import os
with tempfile.TemporaryDirectory() as spec_dir:
    spec_path = os.path.join(spec_dir, "multiply.pythran")
    with open(spec_path, "w") as f:
        f.write("export multiply(float, float)\nexport add(int, int)\n")
    file_specs = pythran.load_specfile(spec_path)
print(f"  Functions in multiply.pythran: {sorted(file_specs.functions)}")
assert file_specs.functions == {"multiply": ((float, float),), "add": ((int, int),)}, file_specs.functions

print("\nspec_parser extracts type signatures for compilation")

In [ ]:
# Example 5: Using pythran.test_compile() and Compilation Testing
import pythran
import tempfile
import sys

print("Using pythran.test_compile() and Compilation Testing:")
print("=" * 55)

# test_compile() runs a comprehensive compilation test
# It's useful for verifying the Pythran installation

print("Running pythran.test_compile()...")
print("This tests the full compilation pipeline\n")

# test_compile() has no return value (it returns None on success) and RAISES
# pythran.errors.PythranCompileError when the C++ toolchain cannot build a module -- so the call itself is the check
pythran.test_compile()
print("✓ Compilation test PASSED")
print("  Pythran is correctly installed and configured")
print("  C++ compiler is available and working")
print("  All necessary headers are accessible")

# One real round-trip: compile an exported function to an extension module, import it, call it
import importlib.util
import os
round_trip_code = '''
#pythran export scaled_sum(float list, float)
def scaled_sum(values, k):
    return k * sum(values)
'''
with tempfile.TemporaryDirectory() as tmpdir:
    so_path = pythran.compile_pythrancode("scaled_mod", round_trip_code, output_file=os.path.join(tmpdir, "scaled_mod.so"))
    spec = importlib.util.spec_from_file_location("scaled_mod", so_path)
    scaled_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(scaled_mod)
    total = scaled_mod.scaled_sum([1.0, 2.0, 3.5], 2.0)
assert total == 13.0, total
print(f"✓ Compiled and imported scaled_mod.scaled_sum([1.0, 2.0, 3.5], 2.0) = {total}")

In [ ]:
# Example 5 (continued): setuptools integration and compilation utilities
import pythran

# Demonstrate PythranExtension for setuptools integration
print("\n" + "="*55)
print("Pythran setuptools integration:")

if hasattr(pythran, 'PythranExtension'):
    print("✓ PythranExtension class available")
    print("  Used in setup.py for building extensions")
    
    # Show how it would be used
    example_usage = '''
from pythran import PythranExtension

# In setup.py:
ext_modules = [
    PythranExtension(
        "my_module",
        sources=["my_module.py"]
    )
]

setup(
    name="my_package",
    ext_modules=ext_modules,
    ...
)
'''
    print("\n  Example usage:")
    for line in example_usage.strip().split('\n')[:10]:
        print(f"    {line}")
else:
    raise AssertionError("pythran.PythranExtension is missing")

# Check compilation utilities
print("\n" + "="*55)
print("Compilation utilities available:")

utilities = {
    'compile_pythrancode': 'Compile Python code string to module',
    'compile_pythranfile': 'Compile Python file to module',
    'compile_cxxcode': 'Compile C++ code to module',
    'compile_cxxfile': 'Compile C++ file to module'
}

for func_name, description in utilities.items():
    if hasattr(pythran, func_name):
        print(f"  ✓ {func_name}: {description}")
    else:
        raise AssertionError(f"pythran.{func_name} is missing")

# Check optimization capabilities
print("\n" + "="*55)
print("Optimization and analysis modules:")

modules = ['optimizations', 'analyses', 'passmanager', 'transformations']
for mod_name in modules:
    if hasattr(pythran, mod_name):
        module = getattr(pythran, mod_name)
        items = [attr for attr in dir(module) if not attr.startswith('_')]
        print(f"  ✓ {mod_name}: {len(items)} items")
    else:
        raise AssertionError(f"pythran.{mod_name} is missing")

print("\n" + "="*55)
print("Summary:")
print("  Pythran provides comprehensive Python-to-C++ compilation")
print("  Key functions: generate_cxx, compile_pythrancode, spec_parser")
print("  Integration: PythranExtension for setuptools, get_include for builds")
print("  Testing: test_compile validates installation")

print("\nPythran enables high-performance scientific Python")